In [1]:
!pip install delta-spark


In [2]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .appName("Part_7_delta") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [3]:
products_data = [
(101,"Rice Bag","Groceries","Hyderabad",1200,50),
(102,"Wheat Flour","Groceries","Bengaluru",900,80),
(103,"Sunflower Oil","Groceries","Mumbai",1800,40),
(104,"Milk Pack","Dairy","Chennai",60,200),
(105,"Cheese Block","Dairy","Delhi",450,70),
(106,"Soap","Personal Care","Kolkata",120,300),
(107,"Shampoo","Personal Care","Pune",320,150),
(108,"Toothpaste","Personal Care","Ahmedabad",90,250),
(109,"Notebook","Stationery","Hyderabad",75,500),
(110,"Pen Pack","Stationery","Mumbai",110,400),
(111,"LED TV","Electronics","Delhi",45000,15),
(112,"Refrigerator","Electronics","Chennai",38000,10),
(113,"Washing Machine","Electronics","Bengaluru",29000,12),
(114,"Mobile Phone","Electronics","Hyderabad",25000,35),
(115,"Laptop","Electronics","Pune",62000,18),
(116,"Air Conditioner","Electronics","Mumbai",42000,9),
(117,"Mixer Grinder","Home Appliances","Kolkata",3500,45),
(118,"Water Purifier","Home Appliances","Delhi",12000,20),
(119,"Ceiling Fan","Home Appliances","Ahmedabad",2800,60),
(120,"Gas Stove","Home Appliances","Chennai",5500,25)
]
products_columns = [
"product_id",
"product_name",
"category",
"warehouse_city",
"price",
"stock_quantity"
]
products_df = spark.createDataFrame(products_data, products_columns)

In [4]:
suppliers_data = [
(201,"Reddy Traders","Hyderabad","Groceries"),
(202,"Fresh Dairy Ltd","Chennai","Dairy"),
(203,"CarePlus Suppliers","Mumbai","Personal Care"),
(204,"Elite Electronics","Delhi","Electronics"),
(205,"OfficeKart","Bengaluru","Stationery"),
(206,"HomeNeeds Pvt Ltd","Pune","Home Appliances"),
(207,"National Grocers","Ahmedabad","Groceries"),
(208,"Smart Electronics","Kolkata","Electronics"),
(209,"Daily Essentials","Hyderabad","Personal Care"),
(210,"Kitchen World","Chennai","Home Appliances")
]
suppliers_columns = [
"supplier_id",
"supplier_name",
"supplier_city",
"specialization"
]
suppliers_df = spark.createDataFrame(suppliers_data, suppliers_columns)

In [5]:
orders_data = [
(301,101,201,"2024-04-01",20,"Delivered"),
(302,102,201,"2024-04-01",35,"Delivered"),
(303,111,204,"2024-04-02",2,"Delivered"),
(304,114,208,"2024-04-02",5,"Pending"),
(305,115,204,"2024-04-03",3,"Delivered"),
(306,104,202,"2024-04-03",50,"Delivered"),
(307,105,202,"2024-04-04",18,"Cancelled"),
(308,117,206,"2024-04-04",7,"Delivered"),
(309,118,210,"2024-04-05",4,"Pending"),
(310,119,206,"2024-04-05",12,"Delivered"),
(311,120,210,"2024-04-06",6,"Delivered"),
(312,113,204,"2024-04-06",4,"Delivered"),
(313,116,208,"2024-04-07",2,"Pending"),
(314,109,205,"2024-04-07",80,"Delivered"),
(315,110,205,"2024-04-08",120,"Delivered"),
(316,106,203,"2024-04-08",60,"Cancelled"),
(317,107,209,"2024-04-09",25,"Delivered"),
(318,108,203,"2024-04-09",40,"Delivered"),
(319,112,208,"2024-04-10",2,"Pending"),
(320,101,207,"2024-04-10",15,"Delivered")
]
orders_columns = [
"order_id",
"product_id",
"supplier_id",
"order_date",
"quantity",
"order_status"
]
orders_df = spark.createDataFrame(orders_data, orders_columns)

In [6]:
payments_data = [
(401,301,24000,"UPI","Paid"),
(402,302,31500,"Credit Card","Paid"),
(403,303,90000,"Bank Transfer","Paid"),
(404,304,125000,"UPI","Pending"),
(405,305,186000,"Bank Transfer","Paid"),
(406,306,3000,"Cash","Paid"),
(407,307,8100,"UPI","Cancelled"),
(408,308,24500,"Debit Card","Paid"),
(409,309,48000,"UPI","Pending"),
(410,310,33600,"Cash","Paid"),
(411,311,33000,"Credit Card","Paid"),
(412,312,116000,"Bank Transfer","Paid"),
(413,313,84000,"UPI","Pending"),
(414,314,6000,"Cash","Paid"),
(415,315,13200,"UPI","Paid"),
(416,316,7200,"Cash","Cancelled"),
(417,317,8000,"UPI","Paid"),
(418,318,3600,"Debit Card","Paid"),
(419,319,76000,"Bank Transfer","Pending"),
(420,320,18000,"UPI","Paid")
]
payments_columns = [
"payment_id",
"order_id",
"bill_amount",
"payment_mode",
"payment_status"
]
payments_df = spark.createDataFrame(payments_data, payments_columns)

In [21]:
#Exercise 61
final_df = orders_df.join(products_df, "product_id") \
                    .join(suppliers_df, "supplier_id") \
                    .join(payments_df, "order_id")
final_df.write.format("delta").mode('overwrite').save("/tmp/final_delta")
spark.sql(""" create table if not exists final_delta using delta location '/tmp/final_delta' """)
spark.sql("select * from final_delta").show()

+--------+-----------+----------+----------+--------+------------+---------------+---------------+--------------+-----+--------------+------------------+-------------+---------------+----------+-----------+-------------+--------------+
|order_id|supplier_id|product_id|order_date|quantity|order_status|   product_name|       category|warehouse_city|price|stock_quantity|     supplier_name|supplier_city| specialization|payment_id|bill_amount| payment_mode|payment_status|
+--------+-----------+----------+----------+--------+------------+---------------+---------------+--------------+-----+--------------+------------------+-------------+---------------+----------+-----------+-------------+--------------+
|     301|        201|       101|2024-04-01|      20|   Delivered|       Rice Bag|      Groceries|     Hyderabad| 1200|            50|     Reddy Traders|    Hyderabad|      Groceries|       401|      24000|          UPI|          Paid|
|     302|        201|       102|2024-04-01|      35|   

In [15]:
#Exercise 62
spark.sql(""" insert into final_delta values
(321, 201, 103, "2024-04-11", 10, "Delivered", "Sunflower Oil", "Groceries", "Mumbai", 1800, 40,
 "Reddy Traders", "Hyderabad", "Groceries", 421, 18000, "UPI", "Paid"),
(322, 204, 115, "2024-04-11", 2, "Pending", "Laptop", "Electronics", "Pune", 62000, 18,
 "Elite Electronics", "Delhi", "Electronics", 422, 124000, "Bank Transfer", "Pending");""")


DataFrame[]

In [19]:
#Exercise 63
spark.sql(""" update final_delta set order_status = "Delivered"
where order_status = "Pending" and order_id = 322""")

DataFrame[num_affected_rows: bigint]

In [22]:
#Exercise 64
spark.sql(""" update final_delta set payment_status = "Paid"
where payment_status = "Pending" and payment_id = 404 """)

DataFrame[num_affected_rows: bigint]

In [23]:
#Exercise 65
spark.sql(""" delete from final_delta where order_status="Cancelled" """)

DataFrame[num_affected_rows: bigint]

In [ ]:
#Exercise 66
spark.sql(""" create or replace table clean_orders using delta
as select * from final_delta where order_status != "Cancelled"
and payment_status != "Cancelled" """)
spark.sql(" select * from clean_orders ").show()

In [28]:
#Exercise 67
spark.sql("describe history final_delta").show()

+-------+--------------------+------+--------+---------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|           timestamp|userId|userName|operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+--------------------+------+--------+---------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|     13|2026-05-06 08:00:...|  NULL|    NULL|   DELETE|{predicate -> ["(...|NULL|    NULL|     NULL|         12|  Serializable|        false|{numRemovedFiles ...|        NULL|Apache-Spark/4.0....|
|     12|2026-05-06 07:59:...|  NULL|    NULL|   UPDATE|{predicate -> ["(...|NULL|    NULL|     NULL|         11|  Serializable|        false|{numRemovedFiles ...|        NULL|Apache-Spark/4.0....|
|     11|2

In [29]:
#Exercise 68
spark.sql(" select * from final_delta version as of 0 ").show()

+--------+-----------+----------+----------+--------+------------+---------------+---------------+--------------+-----+--------------+------------------+-------------+---------------+----------+-----------+-------------+--------------+
|order_id|supplier_id|product_id|order_date|quantity|order_status|   product_name|       category|warehouse_city|price|stock_quantity|     supplier_name|supplier_city| specialization|payment_id|bill_amount| payment_mode|payment_status|
+--------+-----------+----------+----------+--------+------------+---------------+---------------+--------------+-----+--------------+------------------+-------------+---------------+----------+-----------+-------------+--------------+
|     301|        201|       101|2024-04-01|      20|   Delivered|       Rice Bag|      Groceries|     Hyderabad| 1200|            50|     Reddy Traders|    Hyderabad|      Groceries|       401|      24000|          UPI|          Paid|
|     302|        201|       102|2024-04-01|      35|   

In [31]:
#Exercise 69
spark.sql(""" select count(*) as latest_count from final_delta """).show()

spark.sql(""" select count(*) as old_count  from final_delta version as of 0""").show()

+------------+
|latest_count|
+------------+
|          18|
+------------+

+---------+
|old_count|
+---------+
|       20|
+---------+



In [ ]:
#Exercise 70
spark.sql(" vacuum final_delta retain 24 hours dry run")